# Hipótese: Antecedência do cancelamento x Taxa de detratores

Testa se o intervalo registrado em `ANTECEDENCIA_CANCELAMENTO` está associado à taxa de passageiros detratores (`NPS_PRINCIPAL == -100`).

> **Atenção à unidade:** o dicionário de dados descreve o intervalo em dias, enquanto a interpretação anterior usava horas. Até a confirmação com o parceiro, as faixas e os efeitos abaixo são apresentados em **unidades registradas**, sem convertê-las em horas ou dias.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.width', 150)

## 1. Carregar a base analítica oficial

In [ ]:
# Todos os caminhos derivam da raiz do projeto para funcionar tanto da raiz,
# de notebooks/ quanto do Colab. A hipótese consome a mesma base oficial
# produzida por pre-processamento.ipynb; não repete a integração dos brutos.
candidatas_raiz = [Path.cwd(), *Path.cwd().parents, Path('/content'), Path('/content/projeto')]
caminho_base = next(
    (raiz / 'data' / 'processed' / 'base_analitica.parquet'
     for raiz in candidatas_raiz
     if (raiz / 'data' / 'processed' / 'base_analitica.parquet').exists()),
    None,
)

if caminho_base is None:
    raise FileNotFoundError(
        'Não encontrei data/processed/base_analitica.parquet. '
        'Execute notebooks/pre-processamento.ipynb antes desta análise.'
    )

df = pd.read_parquet(caminho_base)
colunas_necessarias = {'RESPONDENT_ID', 'DETRATOR', 'CANCELAMENTO_VOO',
                       'ANTECEDENCIA_CANCELAMENTO', 'NPS_PRINCIPAL'}
assert colunas_necessarias.issubset(df.columns), 'A base analítica não contém as colunas necessárias.'
assert len(df) == 484_915, (
    f'A base analítica deveria ter 484.915 registros e tem {len(df):,}.'
)
assert df['RESPONDENT_ID'].is_unique, 'RESPONDENT_ID deve estar deduplicado na base analítica.'

print(f'Base analítica carregada: {len(df):,} registros')
print(f'Dimensões da base analítica: {df.shape}')

## 2. Isolar voos cancelados e comparar com voos não cancelados

In [ ]:
taxa_geral = df.groupby('CANCELAMENTO_VOO', observed=False)['DETRATOR'].mean()
print(taxa_geral)

## 3. Segmentar cancelamentos por faixa de antecedência

As faixas abaixo usam os valores brutos da variável. A unidade de medida ainda precisa ser confirmada antes de interpretá-las como horas ou dias.

In [ ]:
cancelados = df[df['CANCELAMENTO_VOO'] == True].copy()

bins = [-1, 0, 6, 24, 48, np.inf]
labels = ['0 unidades', '1-6 unidades', '7-24 unidades', '25-48 unidades', '>48 unidades']
cancelados['FAIXA_ANTECEDENCIA'] = pd.cut(
    cancelados['ANTECEDENCIA_CANCELAMENTO'], bins=bins, labels=labels
)

tabela = cancelados.groupby('FAIXA_ANTECEDENCIA', observed=False)['DETRATOR'].agg(
    pct_detrator='mean', n='size'
)
tabela.index.name = 'FAIXA_ANTECEDENCIA (unidade a confirmar)'
tabela

## 4. Visualizar o gradiente

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
tabela['pct_detrator'].mul(100).plot(kind='bar', ax=ax, color='#1f77b4')

taxa_base = df[df['CANCELAMENTO_VOO'] == False]['DETRATOR'].mean() * 100
ax.axhline(taxa_base, color='red', linestyle='--',
           label=f'Base sem cancelamento ({taxa_base:.1f}%)')

ax.set_ylabel('% Detratores')
ax.set_xlabel('Antecedência registrada (unidade a confirmar)')
ax.set_title('Antecedência registrada x Taxa de detratores')
ax.legend()
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Teste estatístico e tamanho do efeito

In [ ]:
tabela_contingencia = pd.crosstab(
    cancelados['FAIXA_ANTECEDENCIA'], cancelados['DETRATOR'], dropna=True
)
chi2, p, dof, expected = stats.chi2_contingency(tabela_contingencia)
n_associacao = tabela_contingencia.to_numpy().sum()
cramers_v = np.sqrt(
    (chi2 / n_associacao)
    / min(tabela_contingencia.shape[0] - 1, tabela_contingencia.shape[1] - 1)
)
amplitude_pp = (tabela['pct_detrator'].iloc[0] - tabela['pct_detrator'].iloc[-1]) * 100

print(f'chi2 = {chi2:.2f}')
print('p-valor < 0,001' if p < 0.001 else f'p-valor = {p:.3f}')
print(f'graus de liberdade = {dof}')
print(f'V de Cramér = {cramers_v:.3f}')
print(f'Amplitude entre a primeira e a última faixa: {amplitude_pp:.1f} p.p.')

### Escolha da medida de associação

A correlação de Pearson com `NPS_PRINCIPAL` não é reportada: o NPS assume poucos valores discretos e a relação de interesse é a proporção de detratores por faixa. A tabela, o qui-quadrado e o V de Cramér descrevem essa associação de forma diretamente interpretável.

### Por que não reportar um logit linear

O odds ratio por unidade supõe um efeito linear no log-odds. O gradiente observado não tem esse formato: a queda é mais forte nas faixas iniciais e se estabiliza nas faixas finais. Para não resumir uma relação não linear por um único coeficiente pouco representativo, a evidência principal desta hipótese é a comparação por faixas.

## Conclusão

- Na escala original da variável, a taxa de detratores cai conforme aumenta a antecedência registrada (69,2% → 24,6%), aproximando-se da taxa base de voos sem cancelamento (18,2%).
- A diferença entre as faixas extremas é de 44,6 pontos percentuais. Junto ao qui-quadrado (p < 0,001), o V de Cramér reportado na célula anterior quantifica a magnitude da associação.
- A queda se concentra nas faixas iniciais e se estabiliza nas finais; por isso, não reportamos um odds ratio linear por unidade. A análise descreve associação, não causalidade.
- A unidade de `ANTECEDENCIA_CANCELAMENTO` não deve ser chamada de hora ou dia até a validação do dicionário com o parceiro; portanto, recomendações que dependam de uma antecedência específica permanecem pendentes dessa confirmação.